# Pilot 2: local reference images

Use a small local Mucha corpus to make attractive cover or inline images for WeChat articles. Article text is not an input. This notebook keeps image selection and each paid generation call explicit. Pilot 1 remains separate.


In [ ]:
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display
from graphic_design_helper.images import compare_images
from graphic_design_helper.reference_generation import (
    build_request, generate_from_references, load_corpus, search_corpus,
)

# Launch Jupyter from the repository root.
project_root = Path.cwd().resolve()
if not (project_root / "pyproject.toml").exists():
    raise RuntimeError("Launch Jupyter from the repository root.")
load_dotenv(project_root / ".env", override=False)
pilot_dir = project_root / "experiments" / "pilot2"
items = load_corpus(pilot_dir / "corpus")
print(f"Loaded {len(items)} local reference images.")


## Browse references

Search by a visual word or leave the query empty. The corpus was imported from museum records marked CC0. Source pages remain attached to every item.


In [ ]:
query = "花卉"  # Examples: floral, gold, figure, 花卉, 人物
selected_tags = []  # Optional exact tag filters
matches = search_corpus(items, query, selected_tags)
if matches:
    display(compare_images([item["path"] for item in matches],
                           [f"{item['id']} · {item['title']}" for item in matches], width=180))
for item in matches:
    print(item["id"], "|", item["title"], "|", ", ".join(item["tags"]))
    print("  Source:", item["source_url"])


## Choose references and preview

Select one to three IDs from the gallery. A cover and an inline illustration use different pilot image sizes; neither needs to represent the article. Review the exact prompt and sources below before generation.


In [ ]:
selected_ids = [matches[0]["id"]] if matches else []
role = "cover"  # "cover" or "illustration"
mood = "elegant and inviting"
palette = "harmonious muted colors"
motifs = "floral curves and decorative linework"
extra = ""
quality = "medium"

request = build_request(items, selected_ids, role=role, mood=mood,
                        palette=palette, motifs=motifs, extra=extra,
                        quality=quality)
print("Size:", request["settings"]["size"])
print("\nReferences:")
for reference in request["references"]:
    print("-", reference["title"], "|", reference["source_url"], "|", reference["license"])
print("\nExact image prompt:\n")
print(request["prompt"])


## Generate one image

Set `GENERATE` to `True` only after reviewing the request above. This makes one paid image API call and saves a new attempt with its references, prompt, status, and output. Rerun the preview cell after changing your choices.


In [ ]:
GENERATE = False
if GENERATE:
    current = build_request(items, selected_ids, role=role, mood=mood,
                            palette=palette, motifs=motifs, extra=extra,
                            quality=quality)
    if current["review_token"] != request["review_token"]:
        raise ValueError("The selection or direction changed. Rerun the preview cell.")
    result = generate_from_references(request,
                                      approved_token=request["review_token"],
                                      output_dir=pilot_dir / "outputs")
    print("Saved:", result["image_path"])
    display(compare_images([result["image_path"]], ["New image"], width=420))
else:
    print("No generation call made. Review the preview, then set GENERATE = True.")


## Compare saved images

Saved attempts remain separate. Inspect the output at the size and crop used by your publishing editor before posting.


In [ ]:
saved = sorted((pilot_dir / "outputs").glob("attempt_*/image.png"))
if saved:
    display(compare_images(saved[-6:], [path.parent.name for path in saved[-6:]], width=240))
else:
    print("No generated images yet.")
